# 📓 실습 02. FastAPI 비동기 API 엔드포인트 구동 및 통합 테스트

이 실습에서는 앞서 설계한 에이전트를 실시간으로 호출할 수 있는 비동기 FastAPI 마이크로서비스 웹 서버를 구동하고, 클라이언트 요청을 비동기로 송수신하는 테스트 스위트를 실행합니다.

In [ ]:
import asyncio

# 1. FastAPI 서버 코드 자동 생성 (app.py)
api_code = """
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class SensorInput(BaseModel):
    flow_rate: float
    platen_torque: float

@app.post("/predict")
async def predict_failure(data: SensorInput):
    if data.flow_rate <= 80:
        return {"status": "WARNING", "reason": "Nozzle Clogging Detected", "action_code": "SOP-01"}
    return {"status": "OK", "reason": "Normal Status", "action_code": "NONE"}
"""

with open("main.py", "w", encoding="utf-8") as f:
    f.write(api_code.strip())
print("main.py API 서버 파일 빌드 완료")

In [ ]:
# 2. 비동기 Uvicorn 이벤트 루프 및 Mock 클라이언트 요청 시뮬레이션
# (노트북 세션 내에서 API 호출 로직을 모의 테스트)
async def mock_post_request(flow_rate, platen_torque):
    print(f"[Client] POST /predict (flow_rate={flow_rate}, platen_torque={platen_torque}) 송신")
    await asyncio.sleep(0.5) # 비동기 네트워크 딜레이 시뮬레이션
    
    # 서버 예측 로직 실행
    if flow_rate <= 80:
        response = {"status": "WARNING", "reason": "Nozzle Clogging Detected", "action_code": "SOP-01"}
    else:
        response = {"status": "OK", "reason": "Normal Status", "action_code": "NONE"}
        
    print(f"[Server Response] {response}\n")

async def main_test():
    await mock_post_request(72.5, 42.0)
    await mock_post_request(115.0, 39.5)

# 비동기 루프 구동
await main_test()